# TP LangChain — Construire un agent RAG avec Mistral AI

**Durée : 2h30**

Ce TP est divisé en trois parties progressives :

1. **Partie 1** — Le modèle et les prompts : comprendre comment parler à un LLM
2. **Partie 2** — Créer un agent avec mémoire de conversation
3. **Partie 3** — RAG : donner accès à vos documents à l'agent

À la fin, vous aurez toutes les briques pour construire une API RAG (Phase 2).

> **Convention** : les cellules avec `# TODO` sont à compléter. Les autres s'exécutent telles quelles.

## 🔧 Setup

```bash
uv sync
```

`uv sync` crée automatiquement le virtualenv en Python 3.13 et installe toutes les dépendances déclarées dans `pyproject.toml`. Pas besoin de `uv venv` séparé.

Ensuite, copiez `.env.example` → `.env` à la racine du projet et renseignez votre `MISTRAL_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

if not os.getenv("MISTRAL_API_KEY"):
    import getpass
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("MISTRAL_API_KEY : ")

print("Clé API chargée :", "✅" if os.getenv("MISTRAL_API_KEY") else "❌")

---
# Partie 1 — Le Modèle et les Prompts

Avant de créer un agent, il faut comprendre comment fonctionne un **Chat Model**.

Un chat model est différent d'un simple LLM : il reçoit une **liste de messages** structurés (pas une simple chaîne de texte) et retourne un message de réponse.

**Objectifs :**
- Initialiser `ChatMistralAI`
- Comprendre les types de messages (`system`, `human`, `ai`)
- Observer l'impact du **system prompt**

Documentation : https://python.langchain.com/docs/integrations/chat/mistralai

In [ ]:
from langchain_mistralai import ChatMistralAI

# TODO: Initialisez le modèle ChatMistralAI.
#   - model="mistral-small-latest"
#   - temperature=0 (réponses déterministes, utile pour les tests)
# La clé API est lue automatiquement depuis MISTRAL_API_KEY.
llm = ...

## 🚀 Invocation directe

La méthode `.invoke()` est l'interface commune à tous les composants LangChain.
Un chat model accepte :
- une `str` simple (convertie automatiquement en HumanMessage)
- une liste de messages typés

Il retourne un objet `AIMessage` dont l'attribut `.content` contient la réponse.

In [ ]:
# TODO: Invoquez le modèle avec une question simple (string).
# Affichez response.content
response = ...
print(response.content)

## 💬 Les types de messages

Dans le monde des chat models, toute interaction est structurée en **messages** avec un rôle :

| Type | Rôle | Usage |
|---|---|---|
| `SystemMessage` | `system` | Instructions au modèle (ton, règles, contexte) |
| `HumanMessage` | `human` | Question/message de l'utilisateur |
| `AIMessage` | `ai` | Réponse du modèle (ou historique) |

On peut construire manuellement une liste de messages pour simuler une conversation ou donner des instructions précises.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# TODO: Invoquez le modèle avec une liste de messages.
#   - Un SystemMessage qui instruit le modèle de toujours répondre en rimes.
#   - Un HumanMessage avec la question "Qu'est-ce que l'intelligence artificielle ?"
# Observez comment le SystemMessage modifie le style de la réponse.
messages = [...]

response = llm.invoke(messages)
print(response.content)

## 🎭 Impact du System Prompt

Le **system prompt** est l'instruction fondamentale qui définit le comportement du modèle.
C'est le levier le plus puissant pour orienter une IA : ton, langue, règles, persona, contraintes...

Dans LangChain 1.0, on ne définit plus "manuellement" un prompt template complexe.
On passe directement `system_prompt=` à `create_agent` (que vous verrez en Partie 2).

**Expérimentez maintenant** pour comprendre son importance :

In [ ]:
question = "Explique-moi ce qu'est l'apprentissage automatique."

# TODO: Comparez les réponses du modèle selon deux system prompts différents.
#
# Essai 1 : system prompt pour un expert technique
response_expert = llm.invoke([
    SystemMessage("..."),  # TODO: rédigez un system prompt pour un expert
    HumanMessage(question)
])

# Essai 2 : system prompt pour expliquer à un enfant de 10 ans
response_simple = llm.invoke([
    SystemMessage("..."),  # TODO: rédigez un system prompt adapté pour simplifier
    HumanMessage(question)
])

print("=== Réponse Expert ===")
print(response_expert.content)
print("\n=== Réponse Simplifiée ===")
print(response_simple.content)

---
# Partie 2 — Agent avec Mémoire de Conversation

Un **agent** dans LangChain 1.0 est construit avec `create_agent`. Il orchestre le modèle dans une boucle
**ReAct** (Reasoning + Acting) : raisonne → agit (appelle des tools) → observe → raisonne...

Sans tools, `create_agent` se comporte comme un chatbot : il répond directement avec le LLM.
Avec une mémoire (`checkpointer`), il se souvient des échanges précédents pour chaque `thread_id`.

```
create_agent(model, tools=[], system_prompt=..., checkpointer=InMemorySaver())
    → agent.invoke({"messages": [...]}, config={"configurable": {"thread_id": "session-1"}})
```

Documentation : https://python.langchain.com/docs/concepts/agents/

## 🤖 Premier agent (sans mémoire)

In [ ]:
from langchain.agents import create_agent

# TODO: Créez un agent simple sans tools et sans mémoire.
#   - model=llm
#   - tools=[] (pas de tools pour l'instant)
#   - system_prompt="Tu es un assistant concis. Réponds en 2-3 phrases maximum en français."
agent = ...

# TODO: Invoquez l'agent avec une question.
# L'agent attend un dict {"messages": [...]}
# Il retourne un dict {"messages": [...]} où le dernier message est la réponse AI.
result = ...

# Extraction de la réponse : la dernière entrée de la liste de messages
print(result["messages"][-1].content)

## 🧠 Ajout de la mémoire avec `InMemorySaver`

Sans mémoire, l'agent oublie chaque échange. Pour maintenir une conversation cohérente,
on ajoute un **checkpointer** qui sauvegarde l'état (dont l'historique des messages) après chaque invocation.

`InMemorySaver` stocke tout en RAM. En production, on utiliserait un checkpointer persistant (base de données).

Le `thread_id` dans `config` identifie la session. Deux `thread_id` différents = deux conversations indépendantes.

Documentation : https://python.langchain.com/docs/concepts/persistence/

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# TODO: Créez un agent avec mémoire.
#   - Instanciez un InMemorySaver()
#   - Passez-le à create_agent via checkpointer=...
#   - Gardez le même system_prompt qu'avant
checkpointer = ...
agent_with_memory = ...

In [ ]:
# Test : deux questions liées dans la même session (même thread_id)
# L'agent doit "se souvenir" de la première réponse pour répondre à la deuxième.

SESSION_ID = "session-demo"
CONFIG = {"configurable": {"thread_id": SESSION_ID}}

# TODO: Invoquez agent_with_memory avec une première question.
# Choisissez une question dont la réponse contient un élément précis
# (ex: "Quel est le pays le plus peuplé du monde ?")
result1 = ...
print("Réponse 1 :", result1["messages"][-1].content)

# TODO: Invoquez agent_with_memory avec une deuxième question qui fait
# référence à la première réponse (ex: "Et quelle est sa capitale ?")
# Si la mémoire fonctionne, l'agent comprendra le "sa" sans explication.
result2 = ...
print("Réponse 2 :", result2["messages"][-1].content)

In [ ]:
# Vérification de l'isolation des sessions :
# Avec un nouveau thread_id, l'agent ne doit PAS se souvenir de l'échange précédent.

result_new_session = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Et sa capitale, c'est quoi ?"}]},
    config={"configurable": {"thread_id": "nouvelle-session"}}
)
print("Réponse nouvelle session (doit être confuse / demander de préciser) :")
print(result_new_session["messages"][-1].content)

## 📡 Bonus — Streaming (optionnel)

Pour des expériences utilisateur réactives (comme ChatGPT), on peut streamer les tokens
au fil de leur génération avec `.stream()`.

In [ ]:
# TODO (bonus): Utilisez .stream() au lieu de .invoke() pour afficher les tokens
# au fur et à mesure. stream_mode="values" retourne l'état complet à chaque étape.
# Affichez uniquement le contenu du dernier message quand il n'est pas vide.

for step in agent_with_memory.stream(
    {"messages": [{"role": "user", "content": "Raconte-moi une brève histoire sur un robot."}]},
    config={"configurable": {"thread_id": "stream-demo"}},
    stream_mode="values",
):
    last_msg = step["messages"][-1]
    # TODO: affichez last_msg.content si non vide

---
# Partie 3 — RAG : Retrieval-Augmented Generation

Un LLM ne connaît que les données sur lesquelles il a été entraîné. Le **RAG** lui donne accès
à **vos propres documents** au moment où il répond.

**Principe :** on encapsule la recherche documentaire dans un `@tool`. L'agent appelle ce tool
automatiquement quand il a besoin d'informations venant des documents.

```
Question → Agent → @tool retrieve_context → FAISS → Documents → LLM → Réponse
```

**Étapes :**
1. Charger et découper des PDFs
2. Créer la base vectorielle FAISS
3. Définir le tool de retrieval
4. Créer l'agent RAG

## 📂 Préparation : placez vos PDFs

Avant de continuer, placez au moins un fichier PDF dans le dossier `../data/`. Par exemple :
[document d'annonce de Centrale Méditerranée](http://www.groupe-centrale.com/wp-content/uploads/2023/03/DP_Lancement-Centrale-Mediterranee.pdf)

On va commencer par utiliser un loader assez simple `PyPDFDirectoryLoader`  
Documentation : https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/


In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

DATA_FOLDER = "../data/"

# TODO: Instanciez un PyPDFDirectoryLoader pointant vers DATA_FOLDER.
# Chargez les documents avec .load().
loader = ...
docs = ...

print(f"{len(docs)} page(s) chargée(s)")
if docs:
    print("Extrait :", docs[0].page_content[:300])

## ✂️ Découpage en chunks

Les LLMs ont une fenêtre de contexte limitée. On découpe les documents en petits fragments
(**chunks**) avant de les vectoriser. `RecursiveCharacterTextSplitter` respecte les
séparateurs naturels du texte (paragraphes, phrases).

- `chunk_size` : taille max d'un fragment (en caractères)
- `chunk_overlap` : chevauchement entre fragments pour ne pas perdre le contexte aux frontières

Documentation : https://docs.langchain.com/oss/python/integrations/splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: Créez un RecursiveCharacterTextSplitter avec chunk_size=1000, chunk_overlap=200.
# Appelez .split_documents(docs) pour obtenir la liste de chunks.
text_splitter = ...
all_splits = ...

print(f"{len(all_splits)} chunk(s) créé(s)")

## 💾 Base vectorielle FAISS

Chaque chunk est converti en **vecteur** (embedding) par `MistralAIEmbeddings`
et stocké dans FAISS. Lors d'une requête, on calcule la similarité entre le vecteur
de la question et les vecteurs des chunks pour retrouver les plus pertinents.

Documentation FAISS : https://python.langchain.com/docs/integrations/vectorstores/faiss/

In [ ]:
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

# TODO: Instanciez MistralAIEmbeddings avec model="mistral-embed".
embedding_model = ...

# TODO: Créez le vectorstore FAISS avec FAISS.from_documents(all_splits, embedding_model).
vectorstore = ...

# Sauvegarde locale pour la Phase 2 (src/rag_engine.py la rechargera)
vectorstore.save_local("../data/faiss_index")
print("Vectorstore créé et sauvegardé dans ../data/faiss_index")

In [ ]:
# TODO: Testez la base vectorielle avec similarity_search.
# Choisissez une question relative au contenu de vos PDFs.
# Affichez le contenu des 2 premiers résultats.
query = "Votre question ici"
results = ...

for i, doc in enumerate(results):
    print(f"--- Résultat {i+1} ---")
    print(doc.page_content[:400])
    print()

## 🔧 Création du Tool de retrieval

Dans LangChain 1.0, on expose la recherche documentaire comme un **tool** avec le décorateur `@tool`.
L'agent décide lui-même quand appeler ce tool en fonction de la question.

```python
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    ...
    return texte_pour_le_llm, documents_bruts
```

`response_format="content_and_artifact"` permet de retourner **deux choses** :
1. Le texte sérialisé envoyé au LLM (le `content`)
2. Les objets `Document` bruts (l'`artifact`) — utiles pour afficher les sources

**Important** : la docstring du tool devient sa description pour le LLM. Soyez précis !

Documentation : https://docs.langchain.com/oss/python/langchain/tools

In [ ]:
from langchain.tools import tool

# TODO: Créez un tool `retrieve_context` qui :
#   1. Prend une `query: str` en paramètre
#   2. Utilise vectorstore.similarity_search(query, k=3) pour récupérer les 3 docs les plus proches
#   3. Sérialise les documents en texte (source + contenu)
#   4. Retourne (texte_sérialisé, liste_de_docs)
#
# N'oubliez pas :
#   - Le décorateur @tool(response_format="content_and_artifact")
#   - Une docstring claire pour guider le modèle

# TODO: Implémentez le tool ici
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    # TODO
    ...

## 🚀 Agent RAG complet

On assemble maintenant le tout : le `retrieve_context` tool + `create_agent` + mémoire.

L'agent va automatiquement :
1. Recevoir la question
2. Décider d'appeler `retrieve_context` avec la bonne requête de recherche
3. Recevoir le contexte documentaire
4. Formuler une réponse basée sur ce contexte

La mémoire (`InMemorySaver`) permettra à l'agent de maintenir le contexte entre les questions
d'une même conversation.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

RAG_SYSTEM_PROMPT = (
    "Tu es un assistant expert. Tu as accès à un outil de recherche documentaire. "
    "Utilise-le pour répondre aux questions en te basant sur les documents disponibles. "
    "Si la réponse n'est pas dans les documents, dis-le clairement. "
    "Réponds toujours en français."
)

# TODO: Créez l'agent RAG avec :
#   - model=llm
#   - tools=[retrieve_context]
#   - system_prompt=RAG_SYSTEM_PROMPT
#   - checkpointer=InMemorySaver()
rag_agent = ...

In [ ]:
# Test de l'agent RAG
# Adaptez les questions au contenu de vos PDFs

RAG_SESSION = "rag-demo"

# Question 1 : réponse attendue dans le document
result1 = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "En quelle année Centrale Méditerranée a-t-elle été créée ?"}]},
    config={"configurable": {"thread_id": RAG_SESSION}}
)
print("Réponse 1 :", result1["messages"][-1].content)

In [ ]:
# Question 2 : fait référence à la réponse précédente (test de la mémoire)
result2 = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "Et quels sont ses axes de développement prioritaires ?"}]},
    config={"configurable": {"thread_id": RAG_SESSION}}
)
print("Réponse 2 :", result2["messages"][-1].content)

In [ ]:
# Question hors contexte : l'agent doit signaler qu'il n'a pas l'information
result3 = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Marseille aujourd'hui ?"}]},
    config={"configurable": {"thread_id": RAG_SESSION}}
)
print("Réponse hors contexte :", result3["messages"][-1].content)

---
# Phase 2 — Industrialisation avec FastAPI

Vous avez construit un agent RAG fonctionnel en notebook. L'objectif de cette phase est d'**exposer cet agent en API REST** afin qu'il puisse être interrogé depuis n'importe quel client (application web, mobile, autre service...).

## Qu'est-ce qu'une API REST ?

Une **API REST** est un service web que l'on interroge via des requêtes HTTP standard.
Chaque **endpoint** est une URL associée à une action :

| Méthode HTTP | URL | Rôle |
|---|---|---|
| `GET` | `/health` | Vérifier que le serveur est opérationnel |
| `POST` | `/chat` | Envoyer une question et recevoir une réponse |

**FastAPI** (le framework utilisé) génère automatiquement une documentation interactive
sur `/docs` à partir du code Python — pas besoin d'écrire la doc manuellement.

## Instructions

1. **`src/rag_engine.py`** — Recopiez votre tool `retrieve_context` et l'appel `create_agent` de la Partie 3 dans la fonction `get_rag_agent()`
2. **`src/main.py`** — Complétez le bloc `TODO` dans l'endpoint `POST /chat` pour invoquer l'agent
3. Démarrez le serveur depuis la **racine du projet** :
   ```bash
   uvicorn src.main:app --reload
   ```
   > `uvicorn` est le serveur web. `--reload` redémarre automatiquement à chaque modification du code.

4. Testez avec `curl` — un outil en ligne de commande pour envoyer des requêtes HTTP :

   ```bash
   # Vérifie que le serveur tourne et que l'agent est bien chargé
   curl http://localhost:8000/health
   ```

   ```bash
   # Envoie une question au chatbot
   #
   # -X POST                      méthode HTTP (on envoie des données)
   # -H 'Content-Type: ...'       indique que le corps est en JSON
   # -d '{"question": ..., ...}'  corps de la requête : la question + l'identifiant de session
   #
   # Le session_id permet à l'agent de se souvenir des échanges précédents.
   # Deux requêtes avec le même session_id partagent la même mémoire.

   curl -X POST http://localhost:8000/chat \
        -H 'Content-Type: application/json' \
        -d '{"question": "De quoi parle le document ?", "session_id": "test-1"}'
   ```

   ```bash
   # Même session_id → l'agent se souvient de la question précédente
   curl -X POST http://localhost:8000/chat \
        -H 'Content-Type: application/json' \
        -d '{"question": "Peux-tu développer ?", "session_id": "test-1"}'
   ```

5. Explorez la **documentation interactive** générée automatiquement : http://localhost:8000/docs

---

# Phase 3 — Interface Chainlit (bonus)

L'API REST répond du JSON — pratique pour les services, moins pour une démo en direct.
**Chainlit** est une bibliothèque Python qui génère une interface de chat en quelques lignes,
avec affichage des étapes de raisonnement de l'agent (quel tool a été appelé, avec quelle requête).

```bash
chainlit run app.py
```

Ouvrez ensuite http://localhost:8000 dans votre navigateur (penser à fermer le serveur uvicorn pour éviter les conflits sur le même port).

Chaque onglet / session dispose de sa propre mémoire de conversation grâce au `thread_id` 
qui correspond automatiquement à l'identifiant de session Chainlit.

> Le fichier `app.py` à la racine du projet contient l'intégration complète — aucun TODO.